# Proyecto: de la regresión regularizada al portafolio óptimo

**Fundamentos de Econometría (ECO515), Universidad Panamericana. Otoño 2026.**

Las cuatro partes del proyecto parecen temas distintos, pero comparten una sola idea: **todo depende de los valores propios de una matriz**.

| Parte | Matriz | Qué pasa si está mal condicionada |
|---|---|---|
| 1. Ridge, Lasso, Elastic Net | $X^\top X$ | Los coeficientes de MCO se vuelven muy variables |
| 2. Multicolinealidad | $X^\top X$ | VIF enormes y signos que cambian |
| 3. PCA | $\Sigma$ | Pocos componentes explican casi toda la varianza |
| 4. Markowitz | $\Sigma^{-1}$ | Los pesos óptimos se vuelven extremos e inestables |

## Cómo trabajar este notebook

1. **Escribe tu semilla** en la primera celda de código: la matrícula de un integrante. Con ella se eligen tus sectores, así que cada equipo obtiene números distintos.
2. **Corre las celdas en orden.** Donde veas `# TU CÓDIGO AQUÍ`, escribe tu código y **borra la línea `raise NotImplementedError`**.
3. **Revisa las comprobaciones.** Después de cada paso aparecen líneas `✅` o `❌`. Un `✅` indica que el resultado es coherente, no que tu interpretación sea correcta.
4. **Contesta cada `✍️` con tus números.** Una respuesta sin cifras de tu equipo no recibe puntos.
5. **No cambies los nombres de las variables** que ya están escritos: las celdas siguientes los usan.
6. **Documenta tu trabajo.** Agrega una portada, explica cada paso con celdas de texto propias, comenta tu código e interpreta cada tabla y gráfica. La lista completa está en `INSTRUCCIONES.md`.

**Datos.** Rendimientos mensuales de 30 portafolios de industria de EE. UU. (Kenneth R. French Data Library, 2000-01 a 2026-09, en porcentaje). No hace falta internet.

**Instrucciones de entrega, dashboard y rúbrica:** consulta `INSTRUCCIONES.md`. **Deducciones en papel:** están en el enunciado en PDF y se contestan al final de este notebook.

## 0. Preparación

In [ ]:
SEMILLA = None  # Escribe aquí la matrícula de un integrante (número entero)
EQUIPO = "Nombre del equipo"

if SEMILLA is None:
    raise ValueError("Escribe tu matrícula en SEMILLA antes de continuar.")

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import (ElasticNetCV, LassoCV, LinearRegression, RidgeCV,
                                  lasso_path, ridge_regression)
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.formula.api import ols
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})


def comprobar(condicion, mensaje):
    print(("✅ " if condicion else "❌ ") + mensaje)


ARCHIVO = "industrias_ff30_mensual.csv"
candidatas = [Path("datos") / ARCHIVO, Path("../kit_alumnos/datos") / ARCHIVO, Path(ARCHIVO)]
ruta = next((r for r in candidatas if r.exists()), None)
# Sin el archivo local (por ejemplo, en Colab) se descarga del repositorio del proyecto
fuente = ruta if ruta is not None else f"https://raw.githubusercontent.com/IvliCaesar/Proyecto-Econometria-Regularizacion-PCA-Markowitz/main/datos/{ARCHIVO}"
datos = pd.read_csv(fuente, index_col="fecha")
FACTORES = ["Mkt-RF", "SMB", "HML", "RF"]
SECTORES = [c for c in datos.columns if c not in FACTORES]
excesos = datos[SECTORES].sub(datos["RF"], axis=0)  # rendimiento en exceso de la tasa libre

generador = np.random.default_rng(SEMILLA)
OBJETIVO = str(generador.choice(SECTORES))
ACTIVOS = sorted(str(s) for s in generador.choice(SECTORES, size=12, replace=False))

corte = int(0.75 * len(datos))
ENTRENA, PRUEBA = datos.index[:corte], datos.index[corte:]
print(f"Equipo: {EQUIPO} | Semilla: {SEMILLA}")
print(f"Sector objetivo (Parte 1): {OBJETIVO}")
print(f"Activos del portafolio (Parte 4): {ACTIVOS}")
print(f"Entrenamiento: {ENTRENA[0]} a {ENTRENA[-1]} ({len(ENTRENA)} meses)")
print(f"Prueba:        {PRUEBA[0]} a {PRUEBA[-1]} ({len(PRUEBA)} meses)")

# Parte 1. Regresión lineal, Ridge, Lasso y Elastic Net

**Pregunta:** ¿en qué mejora la regularización a MCO, y en qué casos no?

**Caso.** Un fondo quiere **replicar** el rendimiento de tu sector objetivo con los otros 29 sectores. Es una relación *contemporánea*: mide replicación, **no pronóstico**. Por eso una $R^2$ alta aquí no significa que se pueda anticipar el mercado.

| Método | Se minimiza | Efecto |
|---|---|---|
| MCO | $\lVert y-X\beta\rVert^2$ | Insesgado, sin penalización |
| Ridge | $\lVert y-X\beta\rVert^2+\lambda\lVert\beta\rVert_2^2$ | Encoge todos los coeficientes; casi nunca los hace cero |
| Lasso | $\frac{1}{2n}\lVert y-X\beta\rVert^2+\lambda\lVert\beta\rVert_1$ | Encoge y **selecciona**: hace ceros exactos |
| Elastic Net | $\frac{1}{2n}\lVert y-X\beta\rVert^2+\lambda\left[\rho\lVert\beta\rVert_1+\frac{1-\rho}{2}\lVert\beta\rVert_2^2\right]$ | Selecciona y reparte el peso entre variables correlacionadas |

Tres reglas para no invalidar los resultados:

- **Estandariza dentro de un `Pipeline`.** La penalización depende de la escala, y el escalador debe ajustarse solo con los datos de entrenamiento.
- **Elige $\lambda$ solo con entrenamiento** (validación cruzada). El periodo de prueba se usa una vez, al final.
- **Los valores de `alpha` de Ridge y de Lasso no son comparables.** En `scikit-learn`, las fórmulas de la tabla usan escalas distintas (Lasso divide entre $2n$).

Usamos `KFold` sin barajar porque estimamos una relación contemporánea entre rendimientos con poca autocorrelación. Para **pronosticar** habría que usar `TimeSeriesSplit`.

### Paso 1.1. Separar los datos y estimar MCO

Construye `X_tr`, `y_tr`, `X_te`, `y_te`: la $y$ es el exceso de `OBJETIVO` y la $X$ son los excesos de los otros 29 sectores. Después estima MCO con `statsmodels` (recuerda `sm.add_constant`) y guárdalo en `mco`.

In [ ]:
OTROS = [s for s in SECTORES if s != OBJETIVO]
# TU CÓDIGO AQUÍ: define X_tr, y_tr, X_te, y_te y estima mco = sm.OLS(...).fit()
raise NotImplementedError("Completa este paso y borra esta línea")
print(f"R² = {mco.rsquared:.3f} | R² ajustada = {mco.rsquared_adj:.3f} | "
      f"F = {mco.fvalue:.1f} (p = {mco.f_pvalue:.2g})")
print(f"Coeficientes individuales significativos al 5%: {(mco.pvalues.drop('const') < 0.05).sum()} de {len(OTROS)}")
comprobar(X_tr.shape == (len(ENTRENA), 29), "X_tr tiene 29 regresoras")
comprobar(OBJETIVO not in X_tr.columns, "El sector objetivo no está entre las regresoras")
comprobar(len(mco.params) == 30, "MCO tiene 29 pendientes y una constante")

### Paso 1.1b. Ridge por dentro

En el enunciado demuestras en papel que, si $X=UDV^\top$ es la descomposición en valores singulares,

$$\hat\beta_{\text{Ridge}}=(X^\top X+\lambda I)^{-1}X^\top y=V\,\mathrm{diag}\!\left(\frac{d_j}{d_j^2+\lambda}\right)U^\top y.$$

Comprueba ambas fórmulas con $\lambda=100$, usando `X_est` (regresoras estandarizadas) y `y_cen` (respuesta centrada). Calcula `beta_cerrada`, `beta_svd` y `factor`, el vector de encogimiento $d_j^2/(d_j^2+\lambda)$.

In [ ]:
X_est = StandardScaler().fit_transform(X_tr)
y_cen = (y_tr - y_tr.mean()).to_numpy()
LAMBDA_PRUEBA = 100.0
# TU CÓDIGO AQUÍ: calcula beta_cerrada con np.linalg.solve, y beta_svd y factor con np.linalg.svd(X_est, full_matrices=False)
raise NotImplementedError("Completa este paso y borra esta línea")
print(f"Encogimiento: dirección de mayor varianza {factor.max():.3f}; de menor varianza {factor.min():.3f}")
comprobar(np.allclose(beta_cerrada, ridge_regression(X_est, y_cen, alpha=LAMBDA_PRUEBA)), "La fórmula cerrada coincide con scikit-learn")
comprobar(np.allclose(beta_cerrada, beta_svd), "La fórmula por valores singulares coincide")
comprobar(np.all((factor > 0) & (factor < 1)), "Cada dirección se encoge por un factor entre 0 y 1")

### Paso 1.2. Ajustar los cuatro modelos

Completa `construir_modelos()`. Cada modelo debe ser `make_pipeline(StandardScaler(), estimador)`. Usa:

- `LinearRegression()`
- `RidgeCV(alphas=ALFAS_RIDGE, cv=CV)`
- `LassoCV(alphas=ALFAS_L1, cv=CV, max_iter=20000)`
- `ElasticNetCV(l1_ratio=RAZONES_L1, alphas=ALFAS_L1, cv=CV, max_iter=20000)`

In [ ]:
RAZONES_L1 = [0.1, 0.5, 0.9]
CV = KFold(n_splits=5, shuffle=False)


def construir_modelos(n_alfas=40):
    ALFAS_RIDGE = np.logspace(-2, 4, n_alfas)
    ALFAS_L1 = np.logspace(-3, 1, n_alfas)
    # TU CÓDIGO AQUÍ: devuelve un diccionario con las llaves "MCO", "Ridge", "Lasso" y "Elastic Net"
    raise NotImplementedError("Completa este paso y borra esta línea")


def evaluar(X_a, y_a, X_b, y_b):
    # Ajusta con (X_a, y_a) y evalúa fuera de muestra con (X_b, y_b)
    filas = {}
    for nombre, modelo in construir_modelos().items():
        modelo.fit(X_a, y_a)
        final = modelo[-1]
        error_b = y_b - modelo.predict(X_b)
        filas[nombre] = {
            "RMSE entrenamiento": np.sqrt(np.mean((y_a - modelo.predict(X_a)) ** 2)),
            "RMSE prueba": np.sqrt(np.mean(error_b ** 2)),
            "R² prueba": 1 - np.sum(error_b ** 2) / np.sum((y_b - y_a.mean()) ** 2),
            "Coef. distintos de 0": int(np.sum(np.abs(final.coef_) > 1e-8)),
            "Norma L2 (coef. estand.)": np.linalg.norm(final.coef_),
            "lambda": getattr(final, "alpha_", np.nan),
        }
    return pd.DataFrame(filas).T


tabla_completa = evaluar(X_tr, y_tr, X_te, y_te)
display(tabla_completa)
comprobar(tabla_completa.loc["Ridge", "Norma L2 (coef. estand.)"] <= tabla_completa.loc["MCO", "Norma L2 (coef. estand.)"],
          "Ridge encoge la norma respecto de MCO")
comprobar(tabla_completa.loc["MCO", "RMSE entrenamiento"] == tabla_completa["RMSE entrenamiento"].min(),
          "MCO tiene el menor error de ENTRENAMIENTO, como debe ser")

### Paso 1.3. Trayectorias de los coeficientes

Grafica cómo cambian los coeficientes con $\lambda$. Ya está hecho Ridge; completa Lasso con `lasso_path(X, y, alphas=malla_l1)`, que devuelve `(alphas, coefs, _)` con los coeficientes en las **filas** de `coefs`.

In [ ]:
malla_ridge = np.logspace(-1, 5, 80)
malla_l1 = np.logspace(-3, 1, 80)
trayectoria_ridge = np.array([ridge_regression(X_est, y_cen, alpha=a) for a in malla_ridge])
# TU CÓDIGO AQUÍ: calcula trayectoria_lasso con forma (n_alphas, 29), en el orden de malla_l1
raise NotImplementedError("Completa este paso y borra esta línea")

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
ejes[0].plot(malla_ridge, trayectoria_ridge, lw=1)
ejes[0].set(xscale="log", title="Ridge: los coeficientes se encogen", xlabel="lambda", ylabel="Coeficiente estandarizado")
ejes[1].plot(malla_l1, trayectoria_lasso, lw=1)
ejes[1].set(xscale="log", title="Lasso: los coeficientes llegan a cero", xlabel="lambda")
plt.tight_layout(); plt.show()
comprobar(trayectoria_lasso.shape == (80, 29), "La trayectoria de Lasso tiene la forma correcta")
comprobar(np.abs(trayectoria_lasso[-1]).sum() < np.abs(trayectoria_lasso[0]).sum(),
          "Al crecer lambda, Lasso encoge los coeficientes")

### Paso 1.4. ¿Y si hay pocos datos?

Repite la comparación con ventanas de entrenamiento cada vez más cortas: los **últimos** $n$ meses de entrenamiento, para $n\in\{36, 60, 120,\ \text{todos}\}$. El periodo de prueba no cambia. Guarda en `rmse_ventanas` un `DataFrame` con una columna por modelo, una fila por $n$ y el RMSE de prueba.

In [ ]:
VENTANAS = [36, 60, 120, len(X_tr)]
# TU CÓDIGO AQUÍ: para cada n usa X_tr.iloc[-n:], y_tr.iloc[-n:] y la función evaluar
raise NotImplementedError("Completa este paso y borra esta línea")
rmse_ventanas.index.name = "n de entrenamiento"
display(rmse_ventanas)
ax = rmse_ventanas.plot(marker="o", title=f"RMSE fuera de muestra: replicar {OBJETIVO}")
ax.set(xlabel="Meses de entrenamiento", ylabel="RMSE de prueba (%)"); plt.show()
comprobar(list(rmse_ventanas.index) == VENTANAS and rmse_ventanas.shape[1] == 4, "Una fila por ventana y una columna por modelo")

### Paso 1.4b. ¿La diferencia es estadísticamente significativa?

Un RMSE menor puede deberse al azar. La prueba de **Diebold-Mariano** contrasta si dos modelos tienen la misma pérdida esperada: con $d_t=e_{\text{MCO},t}^2-e_{\text{Ridge},t}^2$, se regresa $d_t$ sobre una constante con errores estándar HAC.

Ajusta MCO y Ridge con **todo** el entrenamiento y calcula `d`. Guarda en `dm` el resultado de `sm.OLS(d, np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": 3})`.

In [ ]:
pred_mco = construir_modelos()["MCO"].fit(X_tr, y_tr).predict(X_te)
pred_ridge = construir_modelos()["Ridge"].fit(X_tr, y_tr).predict(X_te)
# TU CÓDIGO AQUÍ: calcula d (pérdida de MCO menos pérdida de Ridge) y dm
raise NotImplementedError("Completa este paso y borra esta línea")
print(f"Diferencia media de pérdida: {dm.params[0]:.3f}; t = {dm.tvalues[0]:.2f}; p = {dm.pvalues[0]:.3f}")
comprobar(abs(dm.params[0] - (tabla_completa.loc["MCO", "RMSE prueba"] ** 2 - tabla_completa.loc["Ridge", "RMSE prueba"] ** 2)) < 1e-8,
          "La media de d es la diferencia de los errores cuadráticos medios")

✍️ **Pregunta 1.1.** Con tus números: ¿cuántos sectores conserva Lasso de los 29?, ¿cuál método gana con muestra completa y cuál con $n=36$?, ¿por cuánto, en términos porcentuales? ¿La diferencia entre MCO y Ridge con muestra completa es significativa según Diebold-Mariano? ¿Por qué una sola ventana no basta para concluir?

**✍️ Tu respuesta (con los números de TU equipo):**

*Escribe aquí.*

### Paso 1.5. Experimento de Monte Carlo

Para saber **en qué casos** mejora la regularización, simulamos datos donde conocemos el verdadero $\beta$ y variamos tres condiciones:

- **Colinealidad:** $\rho = 0.1$ o $\rho = 0.95$ entre todas las regresoras.
- **Verdad:** *escasa* (3 coeficientes distintos de cero) o *densa* (todos distintos de cero), con la misma $\lVert\beta\rVert$.
- **Datos:** *abundantes* ($n=200,\ p=20$) o *escasos* ($n=40,\ p=30$).

Son 8 escenarios y 4 métodos, con **una muestra independiente para cada método y réplica**. Eso importa: MANOVA supone observaciones independientes. Medimos dos respuestas: el logaritmo del RMSE de prueba y el logaritmo del error de estimación $\lVert\hat\beta-\beta\rVert$.

Completa `simular_datos`: genera $X_{ij}=\sqrt{\rho}\,z_i+\sqrt{1-\rho}\,e_{ij}$, con $z_i$ y $e_{ij}$ normales estándar independientes, y $y=X\beta+\sigma\varepsilon$. La celda tarda unos minutos la primera vez; el resultado queda guardado en un archivo.

In [ ]:
N_REPETICIONES = 20
METODOS = ["MCO", "Ridge", "Lasso", "Elastic Net"]
ESCENARIOS = [(r, v, d) for r in (0.1, 0.95) for v in ("escasa", "densa") for d in ("abundantes", "escasos")]
TAMANOS = {"abundantes": (200, 20), "escasos": (40, 30)}
SIGMA = 1.5


def simular_datos(n, p, rho, verdad, rng, sigma=SIGMA):
    # Regresa X (n x p), y (n,), beta (p,)
    k = 3
    beta = np.zeros(p)
    if verdad == "escasa":
        beta[:k] = rng.choice([-1.0, 1.0], size=k)
    else:
        beta = rng.normal(size=p)
        beta *= np.sqrt(k) / np.linalg.norm(beta)  # misma norma en ambos casos
    # TU CÓDIGO AQUÍ: genera X e y siguiendo las fórmulas del enunciado
    raise NotImplementedError("Completa este paso y borra esta línea")
    return X, y, beta


def correr_monte_carlo(semilla, repeticiones=N_REPETICIONES):
    rng = np.random.default_rng(semilla)
    filas = []
    for rho, verdad, disp in ESCENARIOS:
        n, p = TAMANOS[disp]
        for metodo in METODOS:
            for _ in range(repeticiones):
                X, y, beta = simular_datos(n + 500, p, rho, verdad, rng)
                modelo = construir_modelos(n_alfas=10)[metodo].fit(X[:n], y[:n])  # malla corta: más rápido
                error = y[n:] - modelo.predict(X[n:])
                coef = modelo[-1].coef_ / modelo[0].scale_  # de regreso a la escala original
                filas.append({
                    "metodo": metodo, "colinealidad": f"rho={rho}", "verdad": verdad, "datos": disp,
                    "escenario": f"rho={rho}|{verdad}|{disp}",
                    "log_rmse": np.log(np.sqrt(np.mean(error ** 2))),
                    "log_error_beta": np.log(np.linalg.norm(coef - beta)),
                })
    return pd.DataFrame(filas)


# Se guarda en disco para no repetirlo. Si cambias simular_datos, borra el archivo.
archivo_mc = Path(f"montecarlo_{SEMILLA}.csv")
if archivo_mc.exists():
    mc = pd.read_csv(archivo_mc)
else:
    mc = correr_monte_carlo(SEMILLA)
    mc.to_csv(archivo_mc, index=False)
_X, _y, _b = simular_datos(100_000, 3, 0.95, "escasa", np.random.default_rng(1))
comprobar(abs(np.corrcoef(_X[:, 0], _X[:, 1])[0, 1] - 0.95) < 0.01, "La correlación simulada entre regresoras es rho")
comprobar(len(mc) == 8 * 4 * N_REPETICIONES, "El diseño está balanceado")
tabla_rmse = mc.pivot_table(index=["colinealidad", "verdad", "datos"], columns="metodo", values="log_rmse", aggfunc="mean").pipe(np.exp)[METODOS]
tabla_beta = mc.pivot_table(index=["colinealidad", "verdad", "datos"], columns="metodo", values="log_error_beta", aggfunc="mean").pipe(np.exp)[METODOS]
print("RMSE de prueba (media geométrica). El mínimo posible es sigma = 1.5")
display(tabla_rmse.style.format("{:.3f}").highlight_min(axis=1, color="#b7e4c7"))
print("Error de estimación de beta (media geométrica)")
display(tabla_beta.style.format("{:.3f}").highlight_min(axis=1, color="#b7e4c7"))

### Paso 1.6. Tabla MANOVA, construida a mano

Queremos contrastar si el **vector** de respuestas $(\log\text{RMSE},\ \log\lVert\hat\beta-\beta\rVert)$ depende del método, del escenario y de su interacción. Para un diseño balanceado de dos factores ($a$ métodos, $b$ escenarios, $r$ réplicas), con $\bar y_{i\cdot}$, $\bar y_{\cdot j}$, $\bar y_{ij}$ y $\bar y$ como vectores de medias:

$$H_A=br\sum_i(\bar y_{i\cdot}-\bar y)(\bar y_{i\cdot}-\bar y)^\top,\qquad H_B=ar\sum_j(\bar y_{\cdot j}-\bar y)(\bar y_{\cdot j}-\bar y)^\top$$

$$H_{AB}=r\sum_{i,j}(\bar y_{ij}-\bar y_{i\cdot}-\bar y_{\cdot j}+\bar y)(\cdot)^\top,\qquad E=\sum_{i,j,k}(y_{ijk}-\bar y_{ij})(\cdot)^\top$$

Si $\theta_1,\dots,\theta_s$ son los valores propios de $E^{-1}H$, los cuatro estadísticos son:

$$\Lambda_{\text{Wilks}}=\prod_i\frac{1}{1+\theta_i},\quad V_{\text{Pillai}}=\sum_i\frac{\theta_i}{1+\theta_i},\quad T_{\text{Hotelling-Lawley}}=\sum_i\theta_i,\quad R_{\text{Roy}}=\max_i\theta_i$$

Completa `matrices_manova` y `estadisticos`. Recuerda: **MANOVA no diagnostica multicolinealidad**. Compara medias de varias respuestas entre grupos; la multicolinealidad es una relación entre las columnas de $X$.

In [ ]:
RESPUESTAS = ["log_rmse", "log_error_beta"]


def matrices_manova(df, factor_a, factor_b, respuestas=RESPUESTAS):
    a, b = df[factor_a].nunique(), df[factor_b].nunique()
    r = len(df) / (a * b)
    m = df[respuestas].mean().to_numpy()
    m_a = df.groupby(factor_a)[respuestas].mean()
    m_b = df.groupby(factor_b)[respuestas].mean()
    m_ab = df.groupby([factor_a, factor_b])[respuestas].mean()
    # TU CÓDIGO AQUÍ: calcula H_a, H_b, H_ab y E con las fórmulas del enunciado
    raise NotImplementedError("Completa este paso y borra esta línea")
    return {factor_a: H_a, factor_b: H_b, f"{factor_a} x {factor_b}": H_ab}, E


def estadisticos(H, E):
    # TU CÓDIGO AQUÍ: obtén los valores propios de E^-1 H (parte real) y calcula los cuatro estadísticos
    raise NotImplementedError("Completa este paso y borra esta línea")


H_dict, E = matrices_manova(mc, "metodo", "escenario")
a_mano = pd.DataFrame({efecto: estadisticos(H, E) for efecto, H in H_dict.items()}).T
print("Estadísticos calculados a mano")
display(a_mano)
T = (mc[RESPUESTAS] - mc[RESPUESTAS].mean()).pipe(lambda d: d.T @ d).to_numpy()
comprobar(np.allclose(sum(H_dict.values()) + E, T),
          "Descomposición exacta: H_metodo + H_escenario + H_interaccion + E = T")

### Paso 1.7. Tabla MANOVA con `statsmodels` y tamaño de efecto

Compara tu cálculo con `statsmodels`. La fórmula usa contrastes de suma, `C(metodo, Sum)`, para que cada término contraste el efecto principal correcto. La celda arma la tabla que reportarás.

Agregamos el **tamaño de efecto** $\eta^2_{\text{multiv.}}=1-\Lambda^{1/s}$, con $s=\min(\text{respuestas},\ gl_H)$: con muchas réplicas casi cualquier diferencia es significativa, y un valor $p$ pequeño no dice que la diferencia sea grande.

In [ ]:
resultado = MANOVA.from_formula(
    "log_rmse + log_error_beta ~ C(metodo, Sum) * C(escenario, Sum)", data=mc
).mv_test()

TERMINOS = {
    "C(metodo, Sum)": "metodo",
    "C(escenario, Sum)": "escenario",
    "C(metodo, Sum):C(escenario, Sum)": "metodo x escenario",
}
NOMBRES = {"Wilks' lambda": "Wilks", "Pillai's trace": "Pillai",
           "Hotelling-Lawley trace": "Hotelling-Lawley", "Roy's greatest root": "Roy"}
filas = []
for termino, efecto in TERMINOS.items():
    for original, corto in NOMBRES.items():
        s = resultado.results[termino]["stat"].loc[original]
        filas.append({"Efecto": efecto, "Estadístico": corto, "Valor": float(s["Value"]),
                      "gl num.": float(s["Num DF"]), "gl den.": float(s["Den DF"]),
                      "F": float(s["F Value"]), "p": float(s["Pr > F"])})
tabla_manova = pd.DataFrame(filas).set_index(["Efecto", "Estadístico"])
gl_h = {"metodo": 3, "escenario": 7, "metodo x escenario": 21}
tabla_manova["eta² multiv."] = [
    1 - tabla_manova.loc[(e, "Wilks"), "Valor"] ** (1 / min(2, gl_h[e])) if est == "Wilks" else np.nan
    for e, est in tabla_manova.index
]
display(tabla_manova.style.format({"Valor": "{:.4f}", "gl num.": "{:.0f}", "gl den.": "{:.0f}", "F": "{:.2f}",
                                   "p": "{:.2e}", "eta² multiv.": "{:.3f}"}, na_rep=""))

comprobar(abs(a_mano.loc["metodo", "Wilks"] - tabla_manova.loc[("metodo", "Wilks"), "Valor"]) < 1e-6,
          "Tu Wilks de método coincide con statsmodels")
comprobar(abs(a_mano.loc["metodo x escenario", "Pillai"] - tabla_manova.loc[("metodo x escenario", "Pillai"), "Valor"]) < 1e-6,
          "Tu Pillai de la interacción coincide con statsmodels")

### Paso 1.8. ¿Se cumplen los supuestos? ¿Qué respuesta explica el efecto?

Un MANOVA significativo no dice *cuál* respuesta ni *qué* método. Completa el primero; el segundo ya está hecho:

1. **ANOVA univariada por respuesta** (`ols` + `anova_lm`, `typ=2`). Como hay dos respuestas, usa corrección de Bonferroni: nivel $0.05/2$.
2. **Varianza de los residuos por celda.** MANOVA supone covarianzas iguales entre celdas; la celda calcula la razón entre la mayor y la menor varianza residual de cada respuesta. Una razón grande indica que los valores $p$ son solo aproximados: el diseño balanceado y la traza de Pillai ayudan, pero no lo resuelven del todo.

In [ ]:
seguimiento = {}
for respuesta in RESPUESTAS:
    # TU CÓDIGO AQUÍ: estima ols(f"{respuesta} ~ C(metodo) * C(escenario)", data=mc).fit() y guarda anova_lm(..., typ=2)
    raise NotImplementedError("Completa este paso y borra esta línea")
    print(f"ANOVA de {respuesta} (nivel con Bonferroni: 0.025)")
    display(seguimiento[respuesta])

residuos = mc[RESPUESTAS] - mc.groupby(["metodo", "escenario"])[RESPUESTAS].transform("mean")
var_celda = residuos.groupby([mc["metodo"], mc["escenario"]]).var()
razon_var = var_celda.max() / var_celda.min()
print("Razón varianza máxima / mínima entre celdas:")
print(razon_var.round(2).to_string())
comprobar(set(seguimiento) == set(RESPUESTAS), "Hay una ANOVA por respuesta")

✍️ **Pregunta 1.2. Conclusión de la Parte 1.** Llena esta tabla con tus resultados y justifica cada renglón.

| Situación | ¿Mejora la regularización a MCO en predicción? | ¿Y en la estimación de $\beta$? | Método preferido |
|---|---|---|---|
| Muchos datos, baja colinealidad | | | |
| Muchos datos, alta colinealidad | | | |
| Pocos datos, verdad escasa | | | |
| Pocos datos, verdad densa | | | |

Responde también:

- ¿Qué renglón de la tabla MANOVA responde "*depende del caso*"? ¿Qué dice su $\eta^2$?
- ¿Cuál estadístico reportas como principal y por qué? (Elígelo por el diseño, no por el valor $p$.) ¿Se cumple la igualdad de covarianzas? ¿Qué tanto puedes confiar en los valores $p$?
- ¿Qué pierdes al usar Lasso frente a MCO si tu objetivo es hacer inferencia sobre un coeficiente?

**✍️ Tu respuesta (con los números de TU equipo):**

*Escribe aquí.*

# Parte 2. ¿Cómo evadir el problema de multicolinealidad?

Primero, una precisión: la multicolinealidad **no sesga** a MCO y las pruebas siguen siendo válidas; solo aumenta la varianza de los coeficientes individuales. Si solo te importa predecir y la relación entre regresoras se mantiene, quizá no haya nada que arreglar. Es un problema cuando quieres interpretar un coeficiente individual.

Usaremos **Longley** (16 años, EE. UU.), el mismo ejemplo de las notas del curso, cuyo VIF de GNP es cercano a 1788.

$$\mathrm{VIF}_j=\frac{1}{1-R_j^2}$$

### Paso 2.1. Diagnóstico

Completa `tabla_vif(X)`: agrega la constante a $X$ y usa `variance_inflation_factor` para cada regresora (sin incluir la constante). Calcula también `condicion`: el número de condición de $X$ **estandarizada**, con `np.linalg.cond`.

In [ ]:
longley = sm.datasets.longley.load_pandas().data
y_l = longley["TOTEMP"]
X_l = longley.drop(columns="TOTEMP")


def tabla_vif(X):
    # TU CÓDIGO AQUÍ: devuelve una Serie indexada por nombre de regresora con su VIF
    raise NotImplementedError("Completa este paso y borra esta línea")


def condicion(X):
    # TU CÓDIGO AQUÍ: estandariza X y calcula su número de condición
    raise NotImplementedError("Completa este paso y borra esta línea")


vif_longley = tabla_vif(X_l)
display(vif_longley.to_frame("VIF").assign(Tolerancia=lambda d: 1 / d["VIF"]))
mco_l = sm.OLS(y_l, sm.add_constant(X_l)).fit()
print(f"Número de condición (estandarizado): {condicion(X_l):.1f}")
print(f"R² = {mco_l.rsquared:.4f}; F: p = {mco_l.f_pvalue:.2g}; pendientes significativas al 5%: {(mco_l.pvalues.drop('const') < .05).sum()} de 6")
comprobar(abs(vif_longley["GNP"] - 1788.5) < 1, "El VIF de GNP coincide con las notas (1788.5)")

### Paso 2.2. El síntoma: coeficientes inestables

Quita una observación a la vez, reestima MCO y guarda las pendientes en `jack_coef` (una fila por observación eliminada). La celda reporta cuántas pendientes cambian de signo.

In [ ]:
# TU CÓDIGO AQUÍ: construye jack_coef: DataFrame con las 6 pendientes al omitir cada observación
raise NotImplementedError("Completa este paso y borra esta línea")
resumen_jack = pd.DataFrame({"Completo": mco_l.params.drop("const"), "Mín.": jack_coef.min(), "Máx.": jack_coef.max()})
resumen_jack["Cambia de signo"] = np.sign(resumen_jack["Mín."]) != np.sign(resumen_jack["Máx."])
display(resumen_jack)
comprobar(jack_coef.shape == (16, 6), "16 reestimaciones de 6 pendientes")

### Paso 2.3. Los remedios, comparados

Completa el diccionario `remedios`. Cada entrada es un `(X, modelo)` y se evalúa con error de validación *leave-one-out*, que reajusta todo el `Pipeline` sin la observación omitida.

| Remedio | Idea | Qué completar |
|---|---|---|
| **Quitar variables** | Eliminar las redundantes | $X$ sin `GNP` ni `POP` |
| **Transformar** | Usar razones que tengan sentido económico | `PIB_real_pc` = `GNP`/(`GNPDEFL`·`POP`), más `UNEMP`, `ARMED` |
| **PCR** | Regresar sobre componentes principales | `StandardScaler` + `PCA(n_components=0.95)` + `LinearRegression` |
| **PLS** | Componentes que también miran a $y$ | `StandardScaler` + `PLSRegression(n_components=2)` |
| **Ridge** | Penalizar en lugar de eliminar | `StandardScaler` + `RidgeCV(alphas=np.logspace(-4, 2, 60))` |

**Advertencia.** Longley es una serie de tiempo pequeña y con tendencia: el error LOO solo sirve para comparar remedios en esta muestra. Conseguir más datos también es un remedio, cuando se puede.

In [ ]:
X_transformada = pd.DataFrame({
    "PIB_real_pc": X_l["GNP"] / (X_l["GNPDEFL"] * X_l["POP"]),
    "UNEMP": X_l["UNEMP"],
    "ARMED": X_l["ARMED"],
})
base = make_pipeline(StandardScaler(), LinearRegression())
remedios = {"MCO completo": (X_l, base)}
# TU CÓDIGO AQUÍ: agrega las llaves "Quitar GNP y POP", "Transformar", "PCR", "PLS" y "Ridge"
raise NotImplementedError("Completa este paso y borra esta línea")

filas = {}
for nombre, (X_r, modelo) in remedios.items():
    prediccion = np.ravel(cross_val_predict(modelo, X_r, y_l, cv=LeaveOneOut()))
    usa_columnas = X_r is not X_l or nombre == "MCO completo"
    filas[nombre] = {
        "RMSE LOO": np.sqrt(np.mean((y_l - prediccion) ** 2)),
        "VIF máximo": tabla_vif(X_r).max() if usa_columnas else np.nan,
        "Condición de X": condicion(X_r) if usa_columnas else np.nan,
    }
tabla_remedios = pd.DataFrame(filas).T
display(tabla_remedios)
comprobar(len(remedios) == 6, "Se comparan los seis enfoques")
comprobar(tabla_remedios.loc["Quitar GNP y POP", "VIF máximo"] < tabla_remedios.loc["MCO completo", "VIF máximo"],
          "Quitar variables reduce el VIF")

### Paso 2.4. Un remedio falso: centrar

Muchos textos sugieren centrar las variables. Comprueba cuándo funciona. Calcula el VIF máximo de:

- `X_l` y `X_l` centrada (`X_l - X_l.mean()`);
- $\{x, x^2\}$ con $x=$`UNEMP`, y $\{x-\bar x, (x-\bar x)^2\}$.

In [ ]:
x = X_l["UNEMP"]
# TU CÓDIGO AQUÍ: calcula vif_lineal, vif_lineal_centrado, vif_polinomio, vif_polinomio_centrado (cada uno es el VIF máximo)
raise NotImplementedError("Completa este paso y borra esta línea")
print(f"Regresoras lineales: {vif_lineal:.1f} -> centradas: {vif_lineal_centrado:.1f}")
print(f"Polinomio:           {vif_polinomio:.1f} -> centrado:  {vif_polinomio_centrado:.1f}")
comprobar(abs(vif_lineal - vif_lineal_centrado) < 1e-6, "Centrar NO cambia el VIF de regresoras lineales")
comprobar(vif_polinomio_centrado < vif_polinomio, "Centrar SÍ reduce el VIF de términos polinomiales")

### Paso 2.5. Tus datos: ¿hay multicolinealidad entre sectores?

Calcula el VIF y la condición de `X_tr` (Parte 1). Muestra los cinco VIF mayores.

In [ ]:
# TU CÓDIGO AQUÍ: calcula vif_sectores (Serie ordenada de mayor a menor) y cond_sectores
raise NotImplementedError("Completa este paso y borra esta línea")
display(vif_sectores.head().to_frame("VIF"))
print(f"Número de condición: {cond_sectores:.1f}. En Longley: {condicion(X_l):.1f}")
comprobar(vif_sectores.iloc[0] >= vif_sectores.iloc[-1], "Los VIF están ordenados")

✍️ **Pregunta 2.** Si tu jefe te pide "quitar la multicolinealidad" de Longley:

1. ¿Qué remedio eliges si quiere **interpretar** el efecto de cada variable? ¿Y si solo quiere **predecir**? Justifica con `tabla_remedios`.
2. ¿Por qué PCR y Ridge aparecen sin VIF en la tabla? ¿Qué pierdes al usarlos?
3. Compara la condición de Longley con la de tus sectores. ¿Hay que preocuparse igual en ambos casos?

**✍️ Tu respuesta (con los números de TU equipo):**

*Escribe aquí.*

# Parte 3. PCA a partir de un vector aleatorio

Partimos de un **vector aleatorio** $X\in\mathbb{R}^p$ con media $\mu$ y covarianza $\Sigma$, no de una tabla de datos. El primer componente es la combinación lineal unitaria de mayor varianza:

$$\max_{a}\ \mathrm{Var}(a^\top X)=a^\top\Sigma a\quad\text{sujeto a}\quad a^\top a=1.$$

Con el lagrangiano $\mathcal{L}=a^\top\Sigma a-\lambda(a^\top a-1)$, la condición de primer orden es

$$\Sigma a=\lambda a,$$

es decir, $a$ es un **vector propio** de $\Sigma$ y $\mathrm{Var}(a^\top X)=a^\top\Sigma a=\lambda$. Se maximiza con el mayor valor propio. Los siguientes componentes añaden la restricción de ortogonalidad, y $Y_k=a_k^\top(X-\mu)$ cumple $\mathrm{Var}(Y_k)=\lambda_k$ y $\mathrm{Cov}(Y_j,Y_k)=0$.

Vamos a obtenerlo de tres maneras: con la teoría, con una muestra del vector aleatorio, y con un **vector inicial aleatorio** (método de la potencia).

### Paso 3.1. La población

Esta celda define un $\Sigma$ exacto para tu equipo. Calcula los valores y vectores propios con `np.linalg.eigh` y ordénalos de **mayor a menor**, en `lambdas` y `vectores` (por columnas).

In [ ]:
rng_pca = np.random.default_rng(SEMILLA + 3)
P_PCA = 5
Q, _ = np.linalg.qr(rng_pca.normal(size=(P_PCA, P_PCA)))
LAMBDAS_VERDADERAS = np.array([8.0, 3.0, 1.5, 0.5, 0.2])
Sigma = Q @ np.diag(LAMBDAS_VERDADERAS) @ Q.T
mu = rng_pca.normal(size=P_PCA)

# TU CÓDIGO AQUÍ: calcula lambdas y vectores, ordenados de mayor a menor
raise NotImplementedError("Completa este paso y borra esta línea")
print(pd.DataFrame(Sigma).round(2).to_string())
print("Valores propios:", lambdas.round(3))
print("Varianza explicada acumulada:", (np.cumsum(lambdas) / lambdas.sum()).round(3))
comprobar(np.allclose(lambdas, LAMBDAS_VERDADERAS), "Los valores propios son los correctos y están ordenados")
comprobar(np.allclose(Sigma @ vectores[:, 0], lambdas[0] * vectores[:, 0]), "Se cumple Sigma a = lambda a")
comprobar(np.allclose(vectores.T @ vectores, np.eye(P_PCA)), "Los vectores son ortonormales")

### Paso 3.2. Ninguna dirección aleatoria le gana al primer componente

Sortea 20,000 direcciones unitarias $a$ al azar y calcula $a^\top\Sigma a$. Toda dirección debe dar una varianza entre $\lambda_{\min}$ y $\lambda_{\max}$: este es el cociente de Rayleigh.

In [ ]:
# TU CÓDIGO AQUÍ: genera 20000 direcciones unitarias (una por fila en direcciones) y calcula varianzas = a'Sigma a
raise NotImplementedError("Completa este paso y borra esta línea")
plt.hist(varianzas, bins=60)
plt.axvline(lambdas[0], color="crimson", label=f"lambda_1 = {lambdas[0]:.1f}")
plt.axvline(lambdas[-1], color="seagreen", label=f"lambda_p = {lambdas[-1]:.1f}")
plt.gca().set(title="Varianza de a'X para direcciones al azar", xlabel="a' Sigma a", ylabel="Frecuencia"); plt.legend(); plt.show()
print(f"La mejor de 20,000 direcciones: {varianzas.max():.3f}; lambda_1 = {lambdas[0]:.3f}")
comprobar(varianzas.max() <= lambdas[0] + 1e-9, "Ninguna dirección supera a lambda_1")
comprobar(varianzas.min() >= lambdas[-1] - 1e-9, "Ninguna dirección queda por debajo de lambda_p")

### Paso 3.3. Muestrear el vector aleatorio

Genera $n$ realizaciones de $X\sim N(\mu,\Sigma)$ y calcula la PCA **muestral**, para $n\in\{30,100,1000,10000\}$. Reporta $\hat\lambda_1$ y el ángulo entre $\hat a_1$ y $a_1$. Como $a$ y $-a$ son la misma dirección, usa $\theta=\arccos|\hat a_1^\top a_1|$.

In [ ]:
def angulo(u, v):
    return np.degrees(np.arccos(np.clip(abs(u @ v) / (np.linalg.norm(u) * np.linalg.norm(v)), 0, 1)))


filas = []
for n in [30, 100, 1000, 10_000]:
    # TU CÓDIGO AQUÍ: genera la muestra con rng_pca.multivariate_normal y ajusta PCA(); obtén l1 y a1 de explained_variance_ y components_
    raise NotImplementedError("Completa este paso y borra esta línea")
    filas.append({"n": n, "lambda_1 muestral": l1, "error relativo": abs(l1 / lambdas[0] - 1), "ángulo con a_1 (grados)": angulo(a1, vectores[:, 0])})
convergencia = pd.DataFrame(filas).set_index("n")
display(convergencia)
comprobar(convergencia["ángulo con a_1 (grados)"].iloc[-1] < convergencia["ángulo con a_1 (grados)"].iloc[0],
          "El componente muestral se acerca al poblacional cuando crece n")

Y = (muestra - mu) @ vectores
display(pd.DataFrame(np.cov(Y.T)).round(2))
comprobar(np.allclose(np.cov(Y.T), np.diag(lambdas), atol=0.5), "Los componentes son casi no correlacionados, con Var(Y_k) = lambda_k")

### Paso 3.4. PCA desde un vector inicial aleatorio (método de la potencia)

Ahora, **sin `eigh` ni `sklearn`**. Parte de un vector aleatorio $v_0$ y repite

$$v_{t+1}=\frac{\Sigma v_t}{\lVert\Sigma v_t\rVert}.$$

Cada multiplicación amplifica más la dirección de $\lambda_1$, y el error cae como $(\lambda_2/\lambda_1)^t$. Para el siguiente componente, **deflaciona**: $\Sigma\leftarrow\Sigma-\lambda_1a_1a_1^\top$ y repite.

Completa `metodo_potencia` y `pca_por_potencia`. El valor propio se estima con $\hat\lambda=v^\top\Sigma v$.

In [ ]:
def metodo_potencia(S, rng, iteraciones=200):
    # Regresa (valor propio, vector propio, lista con los vectores de cada iteración)
    # TU CÓDIGO AQUÍ: parte de un vector normal aleatorio de norma 1 y aplica la iteración
    raise NotImplementedError("Completa este paso y borra esta línea")


def pca_por_potencia(S, k, rng, iteraciones=200):
    # Regresa (arreglo de k valores propios, matriz p x k de vectores)
    # TU CÓDIGO AQUÍ: aplica metodo_potencia k veces, deflacionando la matriz después de cada una
    raise NotImplementedError("Completa este paso y borra esta línea")


lam_pot, vec_pot = pca_por_potencia(Sigma, P_PCA, np.random.default_rng(SEMILLA))
display(pd.DataFrame({"Potencia": lam_pot, "eigh": lambdas, "Ángulo (grados)": [angulo(vec_pot[:, k], vectores[:, k]) for k in range(P_PCA)]}))

_, _, historia = metodo_potencia(Sigma, np.random.default_rng(SEMILLA), 40)
errores = np.array([np.radians(max(angulo(v, vectores[:, 0]), 1e-9)) for v in historia])
t = np.arange(len(errores))
teorica = errores[3] * (lambdas[1] / lambdas[0]) ** (t - 3)
plt.semilogy(t, errores, "o-", ms=3, label="Error observado (radianes)")
plt.semilogy(t, teorica, "--", label=f"Teoría: (λ2/λ1)^t = ({lambdas[1]/lambdas[0]:.3f})^t")
plt.gca().set(title="Convergencia del método de la potencia", xlabel="Iteración t", ylim=(1e-9, 3)); plt.legend(); plt.show()
comprobar(np.allclose(lam_pot, lambdas, atol=1e-6), "Los valores propios coinciden con eigh")
comprobar(all(angulo(vec_pot[:, k], vectores[:, k]) < 0.01 for k in range(P_PCA)), "Los vectores coinciden con eigh, salvo el signo")

### Paso 3.5. Datos reales: el factor de mercado

Aplica tu `pca_por_potencia` a la matriz de **correlación** de los 30 sectores (periodo de entrenamiento). Usamos correlación para que un sector volátil, como Coal, no domine solo por su escala. Calcula:

- `var_explicada`: proporción de varianza de cada uno de los 5 componentes;
- `pc1`: serie de $Y_1=a_1^\top Z$, con $Z$ los excesos estandarizados;
- `corr_mercado`: correlación absoluta de `pc1` con `Mkt-RF`.

In [ ]:
Z = (excesos.loc[ENTRENA] - excesos.loc[ENTRENA].mean()) / excesos.loc[ENTRENA].std()
R = Z.corr().to_numpy()
# TU CÓDIGO AQUÍ: calcula lam_R, vec_R (5 componentes), var_explicada, pc1 y corr_mercado
raise NotImplementedError("Completa este paso y borra esta línea")
cargas = pd.Series(vec_R[:, 0] * np.sign(vec_R[:, 0].sum()), index=SECTORES).sort_values()
fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
ejes[0].bar(range(1, 6), var_explicada)
ejes[0].set(title="Varianza explicada", xlabel="Componente", ylabel="Proporción")
cargas.plot.barh(ax=ejes[1], fontsize=7, title="Cargas del primer componente")
plt.tight_layout(); plt.show()
print(f"Varianza explicada: {np.round(var_explicada, 3)}; acumulada a 5: {var_explicada.sum():.3f}")
print(f"Correlación absoluta del primer componente con el mercado: {corr_mercado:.3f}")
comprobar(np.allclose(lam_R, np.sort(np.linalg.eigvalsh(R))[::-1][:5], atol=1e-3), "Tu método coincide con eigvalsh")
comprobar(corr_mercado > 0.8, "El primer componente se parece al mercado")

✍️ **Pregunta 3.**

1. ¿Qué pasa con la velocidad de convergencia del método de la potencia si $\lambda_2/\lambda_1\approx 1$? ¿Y si $v_0$ fuera exactamente ortogonal a $a_1$? ¿Por qué elegirlo al azar resuelve este último problema?
2. ¿Cuántos $n$ se necesitan para que el ángulo baje de 5°? Usa tu tabla `convergencia`.
3. Interpreta económicamente el primer componente de los sectores. ¿Qué implica para la diversificación y para la multicolinealidad de la Parte 2?

**✍️ Tu respuesta (con los números de TU equipo):**

*Escribe aquí.*

# Parte 4. Solución a Markowitz

Con media $\mu$, covarianza $\Sigma$ invertible y pesos $w$, buscamos el portafolio de mínima varianza para un rendimiento $m$:

$$\min_{w}\ \tfrac12w^\top\Sigma w\quad\text{sujeto a}\quad w^\top\mu=m,\quad w^\top\mathbf 1=1.$$

**Lagrangiano y CPO:** $\mathcal{L}=\tfrac12w^\top\Sigma w-\delta(w^\top\mu-m)-\gamma(w^\top\mathbf 1-1)$ da $\Sigma w=\delta\mu+\gamma\mathbf 1$, por lo que

$$w=\delta\,\Sigma^{-1}\mu+\gamma\,\Sigma^{-1}\mathbf 1.$$

Sustituyendo en las restricciones, con

$$A=\mathbf 1^\top\Sigma^{-1}\mu,\quad B=\mu^\top\Sigma^{-1}\mu,\quad C=\mathbf 1^\top\Sigma^{-1}\mathbf 1,\quad D=BC-A^2,$$

se obtiene

$$\delta=\frac{Cm-A}{D},\qquad\gamma=\frac{B-Am}{D},\qquad\sigma^2(m)=\frac{Cm^2-2Am+B}{D}.$$

**Casos especiales:**

| Portafolio | Pesos | Condición |
|---|---|---|
| Mínima varianza global (GMV) | $w_g=\Sigma^{-1}\mathbf 1/C$, con $m_g=A/C$ | Ninguna |
| Tangente (máximo Sharpe) | $w_t=\Sigma^{-1}(\mu-r_f\mathbf 1)/(A-Cr_f)$ | $r_f<A/C$ |

### Paso 4.1. Implementar la solución cerrada

Trabajamos con rendimientos **anualizados** de tus 12 `ACTIVOS`, estimados solo con entrenamiento. Completa `constantes` y `pesos_frontera`, `pesos_gmv` y `pesos_tangente`. Evita `np.linalg.inv`: usa `np.linalg.solve(S, b)`, que es más estable.

In [ ]:
def constantes(mu_v, S):
    # TU CÓDIGO AQUÍ: regresa A, B, C, D
    raise NotImplementedError("Completa este paso y borra esta línea")


def pesos_frontera(m, mu_v, S):
    # TU CÓDIGO AQUÍ: w = delta S^-1 mu + gamma S^-1 1
    raise NotImplementedError("Completa este paso y borra esta línea")


def pesos_gmv(S):
    # TU CÓDIGO AQUÍ: w_g = S^-1 1 / C
    raise NotImplementedError("Completa este paso y borra esta línea")


def pesos_tangente(mu_v, S, rf):
    # TU CÓDIGO AQUÍ: w_t = S^-1 (mu - rf 1) / (A - C rf)
    raise NotImplementedError("Completa este paso y borra esta línea")


def desempeno(w, mu_v, S):
    return w @ mu_v, np.sqrt(w @ S @ w)


rend = datos.loc[ENTRENA, ACTIVOS]
mu_a = rend.mean().to_numpy() * 12
Sigma_a = rend.cov().to_numpy() * 12
rf_a = datos.loc[ENTRENA, "RF"].mean() * 12
A, B, C, D = constantes(mu_a, Sigma_a)
print(f"A={A:.4f}  B={B:.4f}  C={C:.6f}  D={D:.4f}  m_gmv={A/C:.2f}%  rf={rf_a:.2f}%")
print(f"Condición de Sigma: {np.linalg.cond(Sigma_a):.1f}")

m_ejemplo = mu_a.mean()
w_m, w_g, w_t = pesos_frontera(m_ejemplo, mu_a, Sigma_a), pesos_gmv(Sigma_a), pesos_tangente(mu_a, Sigma_a, rf_a)
comprobar(abs(w_m.sum() - 1) < 1e-8 and abs(w_m @ mu_a - m_ejemplo) < 1e-8, "w_m cumple las dos restricciones")
comprobar(abs(w_m @ Sigma_a @ w_m - (C * m_ejemplo ** 2 - 2 * A * m_ejemplo + B) / D) < 1e-6, "La varianza coincide con la fórmula cerrada")
comprobar(rf_a < A / C, "rf < A/C: el portafolio tangente es eficiente")
comprobar(abs(w_t.sum() - 1) < 1e-8, "Los pesos tangentes suman 1")

### Paso 4.2. Comprobar con un optimizador numérico

Verifica la solución cerrada con `scipy.optimize.minimize` (`method="SLSQP"`). Completa `optimizar`, que además admitirá la prohibición de ventas en corto mediante `bounds`.

In [ ]:
def optimizar(m, mu_v, S, en_corto=True):
    n = len(mu_v)
    restricciones = [{"type": "eq", "fun": lambda w: w.sum() - 1}, {"type": "eq", "fun": lambda w: w @ mu_v - m}]
    limites = None if en_corto else [(0, 1)] * n
    # TU CÓDIGO AQUÍ: llama a minimize sobre 0.5 w'Sw desde pesos iguales, con ftol=1e-12 y maxiter=2000, y regresa solucion.x
    raise NotImplementedError("Completa este paso y borra esta línea")


w_num = optimizar(m_ejemplo, mu_a, Sigma_a)
print(f"Máxima diferencia entre pesos, cerrada vs. numérica: {np.abs(w_num - w_m).max():.2e}")
comprobar(np.abs(w_num - w_m).max() < 1e-2, "La solución numérica coincide con la cerrada")

### Paso 4.3. La frontera eficiente

Construye para una malla de rendimientos: `frontera_sigma` (fórmula cerrada) y `sigma_sin_corto` (con `optimizar(..., en_corto=False)`, solo para `malla_sc`, ya que sin ventas en corto el rendimiento no puede salir del rango de los activos). Calcula también `sharpe_t`, la pendiente de la línea del mercado de capitales.

In [ ]:
malla = np.linspace(A / C - 15, mu_a.max() + 15, 200)
malla_sc = np.linspace(mu_a.min() + 1e-3, mu_a.max() - 1e-3, 30)
# TU CÓDIGO AQUÍ: calcula frontera_sigma, sigma_sin_corto y sharpe_t
raise NotImplementedError("Completa este paso y borra esta línea")
m_g, s_g = desempeno(w_g, mu_a, Sigma_a)
m_t, s_t = desempeno(w_t, mu_a, Sigma_a)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(frontera_sigma, malla, label="Frontera con ventas en corto")
ax.plot(sigma_sin_corto, malla_sc, label="Frontera sin ventas en corto")
ax.scatter(np.sqrt(np.diag(Sigma_a)), mu_a, c="gray", s=20, label="Sectores")
for nombre, xx, yy in zip(ACTIVOS, np.sqrt(np.diag(Sigma_a)), mu_a):
    ax.annotate(nombre, (xx, yy), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.scatter([s_g], [m_g], marker="D", s=70, color="seagreen", label="GMV", zorder=3)
ax.scatter([s_t], [m_t], marker="*", s=180, color="crimson", label="Tangente", zorder=3)
x_lmc = np.array([0, frontera_sigma.max()])
ax.plot(x_lmc, rf_a + sharpe_t * x_lmc, "--", color="crimson", lw=1, label="Línea del mercado de capitales")
ax.set(xlabel="Volatilidad anual (%)", ylabel="Rendimiento anual (%)", title="Frontera eficiente (entrenamiento)",
       xlim=(0, max(np.sqrt(np.diag(Sigma_a)).max() * 1.1, s_t * 1.1)), ylim=(malla.min(), malla.max()))
ax.legend(fontsize=8); plt.show()

sigma_cerrada_sc = np.sqrt((C * malla_sc ** 2 - 2 * A * malla_sc + B) / D)
comprobar(np.all(sigma_sin_corto >= sigma_cerrada_sc - 1e-4), "Prohibir ventas en corto nunca reduce el riesgo")
comprobar(abs(sharpe_t - np.sqrt(B - 2 * A * rf_a + C * rf_a ** 2)) < 1e-6, "El Sharpe tangente cumple H = B - 2A rf + C rf²")

### Paso 4.4. Teorema de los dos fondos

**Afirmación:** todo portafolio de la frontera es combinación de dos portafolios cualesquiera de ella. Encuentra el `alfa` tal que $w_m=\alpha w_g+(1-\alpha)w_t$ para tu `m_ejemplo`.

In [ ]:
# TU CÓDIGO AQUÍ: despeja alfa de m = alfa m_g + (1 - alfa) m_t y calcula w_dos_fondos
raise NotImplementedError("Completa este paso y borra esta línea")
print(f"alfa = {alfa:.3f}")
comprobar(np.allclose(w_dos_fondos, w_m, atol=1e-6), "La combinación de dos fondos reproduce el portafolio de la frontera")

### Paso 4.5. La prueba real: fuera de muestra

La frontera de arriba es **ilusoria**: usa los mismos datos con los que se optimizó. Compara las estrategias, estimadas con entrenamiento, en el periodo de **prueba**:

| Estrategia | Pesos |
|---|---|
| `1/N` | Iguales |
| `GMV muestral` | $w_g$ con $\hat\Sigma$ |
| `Tangente muestral` | $w_t$ con $\hat\mu,\hat\Sigma$ |
| `GMV Ledoit-Wolf` | $w_g$ con el estimador encogido `LedoitWolf().fit(rend).covariance_ * 12` |
| `GMV PCA` | $w_g$ con $\hat\Sigma_{\text{PCA}}$, de 3 componentes más la diagonal residual |
| `Tangente sin corto` | Máximo Sharpe con $w\geq 0$ (ya implementado) |

Completa `covarianza_pca` y el diccionario `estrategias`. La celda rebalancea cada mes a los mismos pesos y no incluye costos de transacción. El Sharpe es anualizado y usa exceso sobre `RF`.

In [ ]:
def covarianza_pca(S, k=3):
    # S se aproxima con k componentes; lo que no explican queda como diagonal
    # TU CÓDIGO AQUÍ: S_k = A_k diag(lambda_k) A_k'; regresa S_k + diag(diag(S - S_k))
    raise NotImplementedError("Completa este paso y borra esta línea")


def tangente_sin_corto(mu_v, S, rf):
    n = len(mu_v)
    negativo_sharpe = lambda w: -(w @ mu_v - rf) / np.sqrt(w @ S @ w)
    return minimize(negativo_sharpe, np.ones(n) / n, method="SLSQP", bounds=[(0, 1)] * n,
                    constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], options={"maxiter": 1000}).x


Sigma_lw = LedoitWolf().fit(rend).covariance_ * 12
Sigma_pca = covarianza_pca(Sigma_a)
# TU CÓDIGO AQUÍ: crea estrategias: diccionario nombre -> pesos, con las llaves de la tabla
raise NotImplementedError("Completa este paso y borra esta línea")


def resultado_prueba(w):
    r = datos.loc[PRUEBA, ACTIVOS].to_numpy() @ w
    exceso = r - datos.loc[PRUEBA, "RF"].to_numpy()
    acumulado = np.cumprod(1 + r / 100)
    return {
        "Rendimiento anual (%)": r.mean() * 12,
        "Volatilidad anual (%)": r.std(ddof=1) * np.sqrt(12),
        "Sharpe prueba": exceso.mean() / exceso.std(ddof=1) * np.sqrt(12),
        "Máx. caída (%)": 100 * (1 - acumulado / np.maximum.accumulate(acumulado)).max(),
        "Mayor peso abs.": np.abs(w).max(),
        "Suma pesos cortos": w[w < 0].sum(),
        "Sharpe entrenamiento (optimista)": (w @ mu_a - rf_a) / np.sqrt(w @ Sigma_a @ w),
    }


tabla_prueba = pd.DataFrame({n: resultado_prueba(w) for n, w in estrategias.items()}).T
display(tabla_prueba)
print(f"Condición de Sigma: muestral {np.linalg.cond(Sigma_a):.0f}, Ledoit-Wolf {np.linalg.cond(Sigma_lw):.0f}, PCA {np.linalg.cond(Sigma_pca):.0f}")
comprobar(all(abs(w.sum() - 1) < 1e-6 for w in estrategias.values()), "Todos los pesos suman 1")
comprobar(np.linalg.cond(Sigma_lw) < np.linalg.cond(Sigma_a), "Ledoit-Wolf mejora el condicionamiento (igual que Ridge)")

### Paso 4.6. ¿Qué tan estables son los pesos?

Remuestrea 200 veces los meses de entrenamiento (*bootstrap*) y recalcula los pesos de la tangente y de la GMV. Guarda los pesos en las listas `pesos_t` y `pesos_g`. La celda resume la dispersión con el rango intercuartil, porque la tangente puede dar pesos enormes cuando $A-Cr_f$ se acerca a cero y eso distorsionaría una desviación estándar.

In [ ]:
rng_boot = np.random.default_rng(SEMILLA + 4)
r_ent = rend.to_numpy()
pesos_t, pesos_g = [], []
for _ in range(200):
    # TU CÓDIGO AQUÍ: remuestrea filas de r_ent con reemplazo; estima mu y Sigma anualizados; agrega los pesos a pesos_t y pesos_g
    raise NotImplementedError("Completa este paso y borra esta línea")
pesos_t, pesos_g = np.array(pesos_t), np.array(pesos_g)
iqr = lambda m: np.subtract(*np.percentile(m, [75, 25], axis=0))
inestabilidad = pd.DataFrame({"Tangente (rango intercuartil)": iqr(pesos_t), "GMV (rango intercuartil)": iqr(pesos_g)}, index=ACTIVOS)
display(inestabilidad)
print(f"Rango intercuartil mediano: tangente {np.median(iqr(pesos_t)):.3f}, GMV {np.median(iqr(pesos_g)):.3f}")
comprobar(np.median(iqr(pesos_t)) > np.median(iqr(pesos_g)), "La tangente es más inestable que la GMV: depende de mu, muy difícil de estimar")

### Paso 4.7. Reestimar cada año

Una gestora real no fija los pesos durante años. Al inicio de cada año del periodo de prueba, estima con los **60 meses anteriores** y conserva esos pesos 12 meses, con rebalanceo mensual y sin costos. Completa `prueba_movil(regla)`, donde `regla(mu, S, rf, historia)` devuelve los pesos; `mu`, `S` y `rf` están anualizados y `historia` son los rendimientos de la ventana.

In [ ]:
def prueba_movil(regla, ventana=60):
    rendimientos = []
    for inicio in range(corte, len(datos), 12):
        historia = datos.iloc[inicio - ventana:inicio]
        # TU CÓDIGO AQUÍ: estima mu, S y rf con historia; obtén w = regla(mu, S, rf, historia[ACTIVOS]) y agrega a rendimientos los meses inicio:inicio+12
        raise NotImplementedError("Completa este paso y borra esta línea")
    return np.array(rendimientos)


reglas = {
    "1/N": lambda m, S, r, h: np.ones(len(m)) / len(m),
    "GMV muestral": lambda m, S, r, h: pesos_gmv(S),
    "GMV Ledoit-Wolf": lambda m, S, r, h: pesos_gmv(LedoitWolf().fit(h).covariance_ * 12),
    "Tangente muestral": lambda m, S, r, h: pesos_tangente(m, S, r),
    "Tangente sin corto": lambda m, S, r, h: tangente_sin_corto(m, S, r),
}
rf_prueba = datos.loc[PRUEBA, "RF"].to_numpy()
filas = {}
for nombre, regla in reglas.items():
    r = prueba_movil(regla)
    exceso = r - rf_prueba
    acumulado = np.cumprod(1 + np.maximum(r, -100) / 100)
    filas[nombre] = {"Rendimiento anual (%)": r.mean() * 12, "Volatilidad anual (%)": r.std(ddof=1) * np.sqrt(12),
                     "Sharpe": exceso.mean() / exceso.std(ddof=1) * np.sqrt(12),
                     "Máx. caída (%)": 100 * (1 - acumulado / np.maximum.accumulate(acumulado)).max()}
tabla_movil = pd.DataFrame(filas).T
display(tabla_movil)
comprobar(len(prueba_movil(reglas["1/N"])) == len(PRUEBA), "Hay un rendimiento por cada mes de prueba")

✍️ **Pregunta 4.**

1. ¿Por qué falla el portafolio tangente si $r_f\geq A/C$? ¿Se cumple la condición en tus datos?
2. ¿Cuál estrategia ganó fuera de muestra, medida por Sharpe? Compara "Sharpe entrenamiento" contra "Sharpe prueba" en la tangente muestral. ¿Cuánto se perdió?
3. Explica la analogía: **Ledoit-Wolf es a Markowitz lo que Ridge es a MCO**, y **la covarianza de PCA es a Markowitz lo que PCR es a MCO**.
4. Con este resultado, ¿le recomendarías a un fondo real la solución cerrada de Markowitz? ¿Cambia tu conclusión al reestimar cada año (`tabla_movil`)? ¿Qué limitaciones siguen (un solo periodo, sin costos)?

**✍️ Tu respuesta (con los números de TU equipo):**

*Escribe aquí.*

# Ejercicios en papel

Contesta aquí los ejercicios de deducción del enunciado en PDF: **1.1**, **2.1**, **3.1** y **4.1**. También puedes entregarlos escaneados.

**✍️ Tu respuesta (con los números de TU equipo):**

*Escribe aquí.*

# Parte 5. Dashboard

Sube tus resultados a un dashboard interactivo. Sigue los pasos de `INSTRUCCIONES.md`. **Presentarás este dashboard oralmente, en vivo, ante profesores.** Elige **una** opción:

- **Opción A: Streamlit** (`dashboard/app.py`). Calcula en vivo.
- **Opción B: página HTML** (`dashboard/generar_html.py`). Se publica gratis en GitHub Pages.

Esta celda guarda tus resultados, para que el dashboard pueda mostrarlos sin volver a correr el Monte Carlo.

In [ ]:
carpeta = Path("dashboard/resultados")
carpeta.mkdir(parents=True, exist_ok=True)
tabla_manova.reset_index().to_csv(carpeta / "tabla_manova.csv", index=False)
tabla_rmse.reset_index().to_csv(carpeta / "montecarlo_rmse.csv", index=False)
tabla_beta.reset_index().to_csv(carpeta / "montecarlo_beta.csv", index=False)
tabla_remedios.to_csv(carpeta / "remedios.csv")
tabla_prueba.to_csv(carpeta / "markowitz_prueba.csv")
tabla_movil.to_csv(carpeta / "markowitz_movil.csv")
pd.Series({"SEMILLA": SEMILLA, "EQUIPO": EQUIPO, "OBJETIVO": OBJETIVO, "ACTIVOS": ",".join(ACTIVOS)}).to_json(carpeta / "equipo.json")
print("Resultados guardados en", carpeta.resolve())

# Parte 6. Memorándum final

✍️ En **máximo 250 palabras**, dirigidas al comité de inversión de un fondo, responde: **¿qué método usarías para replicar tu sector y cómo construirías el portafolio?** Incluye tres cifras de tu análisis y una limitación. El comité no sabe qué es un valor propio: explícalo sin esa palabra.

Antes de entregar: revisa que el notebook esté documentado y corre `Kernel > Restart & Run All`. Debe correr de principio a fin sin errores.

**✍️ Tu respuesta (con los números de TU equipo):**

*Escribe aquí.*